In [1]:
import sys

sys.path.append("..")

In [2]:
import numpy as np
import polars as pl
from src.data.config import ROOT_DIR

In [3]:
golden_set = pl.read_parquet(ROOT_DIR / "data" / "golden_set.parquet").filter(pl.col("geonameIds").list.len() > 0)

In [4]:
import httpx
from tqdm.auto import tqdm


def search_batch(queries: list[str], top_k: int = 20):
    base_url = "http://localhost:8000/v1/search"
    results = []
    
    with httpx.Client(timeout=30.0) as client:  # синхронный клиент
        for query in tqdm(queries):
            response = client.get(base_url, params={"query": query, "top_k": top_k})
            response.raise_for_status()
            results.append(response.json())
    
    return results

In [5]:
from ir_measures import P, Recall, RR, calc


qrels_dict = {}
for row in golden_set.iter_rows(named=True):
    qrels_dict.update({row["query"]: {str(gid): 1 for gid in row["geonameIds"]}})

predictions = search_batch(qrels_dict.keys(), top_k=50)

run_dict = {}
for pred in predictions:
    run_dict.update({pred["query"]: {str(r["geoname_id"]): float(np.log10(r["population"])) for r in  pred["results"]}})

metrics = calc([RR, P@1, Recall@5, Recall@25, Recall@50], qrels_dict, run_dict)
metrics_aggregated = pl.DataFrame({str(k): v for k, v in metrics.aggregated.items()}).unpivot().sort("value")

metrics_per_query = {str(m): {"query": [], "value": []} for m in metrics.aggregated}

for metric in metrics.per_query:
    mname = str(metric.measure)
    metrics_per_query[mname]["query"].append(metric.query_id)
    metrics_per_query[mname]["value"].append(metric.value)

for mname in metrics_per_query:
    metrics_per_query[mname] = pl.DataFrame(metrics_per_query[mname]).sort("value")

  0%|          | 0/180 [00:00<?, ?it/s]

In [6]:
metrics_aggregated

variable,value
str,f64
"""P@1""",0.816667
"""R@5""",0.84779
"""RR""",0.882939
"""R@25""",0.95504
"""R@50""",0.987155


In [7]:
for k in metrics_per_query:
    print(k)
    display(metrics_per_query[k].limit(5))

R@25


query,value
str,f64
"""Ağrı Dağı'na tırmanmak için en…",0.0
"""What are the best weekend geta…",0.347826
"""Van, Sinop ve Şanlıurfa’dan bi…",0.5
"""What are the best neighborhood…",0.5
"""Какая погода в Ване в конце ок…",0.5


R@50


query,value
str,f64
"""Ağrı Dağı'na tırmanmak için en…",0.0
"""Van, Sinop ve Şanlıurfa’dan bi…",0.5
"""Rusya'da tur atarken Soçi, Kaz…",0.666667
"""Анталья или Кемер — что лучше …",0.8
"""Где лучше жить: в Портленде ил…",0.846154


RR


query,value
str,f64
"""Ağrı Dağı'na tırmanmak için en…",0.0
"""St. Louis'de Gateway Kemeri'ni…",0.055556
"""Сколько стоит аренда однокомна…",0.1
"""Rostov Velikiy’de bir hafta so…",0.111111
"""Rostov Velikiy'de bir hafta so…",0.111111


P@1


query,value
str,f64
"""Что посмотреть в Сент-Луисе за…",0.0
"""What are some hidden gems to v…",0.0
"""Ağrı'da kış mevsiminde turist …",0.0
"""Какие цены на недвижимость в С…",0.0
"""Какая погода ожидается в Чите …",0.0


R@5


query,value
str,f64
"""Какие цены на недвижимость в С…",0.0
"""St. Louis'de Gateway Kemeri'ni…",0.0
"""Сколько стоит аренда однокомна…",0.0
"""Где лучше всего попробовать св…",0.0
"""Ağrı Dağı'na tırmanmak için en…",0.0


In [8]:
no_city_df = pl.read_parquet(ROOT_DIR / "data" / "golden_set.parquet").filter(pl.col("geonameIds").list.len() == 0)

In [9]:
results = search_batch(no_city_df["query"].to_list())

  0%|          | 0/55 [00:00<?, ?it/s]

In [10]:
results = [r["results"] for r in results]

In [11]:
accuracy = 0
for r in results:
    if r == []:
        accuracy += 1
accuracy /= len(results)

In [12]:
accuracy

0.9272727272727272